# Mølmer–Sørensen two-qubit gate: ideal → errors

Two ions, spin phase $\phi_j = 0$, driven by a bichromatic beat detuned $\delta$ from a motional
sideband. After the Lamb–Dicke expansion to first order **and** the rotating-wave approximation,
the spin-dependent force is

$$ H_{\rm RWA}(t) = \sum_{j} \sigma_{\Phi_j}^{(j)}\left(f_j(t)\,a^\dagger + f_j^*(t)\,a\right),
\qquad f_j(t) = -\frac{\eta\, b_j\, \Omega_j(t)}{2}\,e^{-i\delta t}, \qquad \Phi_j = \phi_j + \tfrac{\pi}{2}. $$

Because all $\sigma_{\Phi_j}$ commute, the Magnus series terminates at second order and the propagator
is **exactly**

$$ U(t) = \exp\Big(\textstyle\sum_j \sigma_{\Phi_j}\big(\alpha_j(t)a^\dagger - \alpha_j^*(t)a\big)\Big)\,
\exp\Big(i\textstyle\sum_{jk}\Theta_{jk}(t)\,\sigma_{\Phi_j}\sigma_{\Phi_k}\Big), $$

$$ \alpha_j(t) = -i\!\int_0^t f_j, \qquad
\Theta_{jk}(t) = \int_0^t\! dt_1\!\int_0^{t_1}\! dt_2\,\mathrm{Im}\!\left[f_j(t_1)f_k^*(t_2)\right]. $$

For a **flat pulse** ($\Omega$ constant) the trajectory is a circle,
$\alpha_j(t) = \frac{c_j}{\delta}(e^{-i\delta t}-1)$ with $c_j = -\eta b_j\Omega/2$, closing at
$T = 2\pi/\delta$, where the entangling angle is $\chi \equiv \Theta_{01}+\Theta_{10} = -4\pi c_0 c_1/\delta^2$.
A maximally entangling gate needs $|\chi| = \pi/4$; with COM participation $b_j = 1/\sqrt2$ that fixes

$$ \eta\,\Omega = \delta/\sqrt{2}. $$

**Code mapping:** `MSMagnus` *is* the boxed $U(t)$ (`.unitary(t)`, consumed by `UnitaryEvolution`
with **no ODE solve** — the analytic-unitary path); `alpha_trajectory` / `geometric_phase` are
$\alpha_j(t)$ / $\Theta_{jk}(t)$; `ms_lamb_dicke1(..., rwa=True)` is $H_{\rm RWA}(t)$ itself, solved
numerically. Numerically, $\alpha$ and $\Theta$ come from dense-grid trapezoid quadrature
(`points_per_period`), not the ODE solver.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt

from htdse import (Operator, HamiltonianEvolution, UnitaryEvolution, quiet,
                   fidelity, otimes, ket, embed)
from htdse.submodules.molmer_sorensen import (MSMagnus, ms_lamb_dicke1,
                                              plot_phase_space, expectation_alpha)
from htdse.submodules.harmonic_oscillator import fock
from htdse.submodules.wigner import wigner   # paste-in module from earlier
from htdse.submodules.spin import sigma_y

# gate parameters
delta = 1.0
eta = 0.1
Omega = delta / (eta * np.sqrt(2))          # |chi| = pi/4 condition
T = 2 * np.pi / delta                        # loop closure
n_max = 12
b_com = [1/np.sqrt(2), 1/np.sqrt(2)]         # COM mode participation
b_str = [1/np.sqrt(2), -1/np.sqrt(2)]        # stretch mode participation
nu_com, nu_str = 20.0, np.sqrt(3) * 20.0     # axial 2-ion mode frequencies
delta_str = (nu_com + delta) - nu_str        # same beat, detuning seen by the stretch mode

mag_com = MSMagnus(b_com, eta, delta, Omega, [0.0, 0.0], n_max)
mag_str = MSMagnus(b_str, eta, delta_str, Omega, [0.0, 0.0], n_max)

Th = mag_com.geometric_phase(T)
chi = Th[0, 1] + Th[1, 0]
print(f"entangling angle chi/pi = {chi/np.pi:+.4f}   (target -1/4)")
print(f"COM loop closure |alpha(T)| = {abs(mag_com.alpha(T)[0]):.2e}")
print(f"stretch residual |alpha(T)| = {abs(mag_str.alpha(T)[0]):.2e}  (fast, small loop)")

## 1. Ideal gate: both motional modes, flat pulse

The two modes see the *same* beat but different detunings ($\delta$ for COM,
$\delta_{\rm str} = \mu - \nu_{\rm str}$ for stretch — large, so its loop is fast and tiny).
Different modes commute and the $\sigma_{\Phi_j}$ are shared, so the two-mode propagator is exactly
the product of the two single-mode Magnus unitaries, each lifted into the joint space with `embed`:

$$ U_{\rm 2mode}(t) = U_{\rm com}(t)\,U_{\rm str}(t), \qquad
\mathcal H = \mathcal H_{q_0}\!\otimes\mathcal H_{q_1}\!\otimes\mathcal H_{\rm com}\!\otimes\mathcal H_{\rm str}. $$

At closure the ideal gate on $|00\rangle$ (mode in vacuum) gives the Bell state
$e^{i\chi\,\sigma_y^{(0)}\sigma_y^{(1)}}|00\rangle = \cos\chi\,|00\rangle - i \sin\chi\,|11\rangle$
(up to a global phase $e^{i(\Theta_{00}+\Theta_{11})}$).

In [ ]:
dims2 = {"q0": 2, "q1": 2, "com": n_max + 1, "str": n_max + 1}
U2 = np.asarray(embed(mag_com.unitary(T), dims2, ("q0", "q1", "com"))) \
   @ np.asarray(embed(mag_str.unitary(T), dims2, ("q0", "q1", "str")))
psi0_2 = otimes(ket("00"), fock(0, n_max), fock(0, n_max))
psi_T = U2 @ psi0_2

bell = otimes((np.cos(chi) * ket("00") - 1j * np.sin(chi) * ket("11")),
              fock(0, n_max), fock(0, n_max))
print(f"Bell fidelity (2 modes): {fidelity(bell, psi_T):.6f}")
print("  (deficit ~ |alpha_str(T)|^2: the stretch loop is open at t=T -- a real,",
      "\n   usually-neglected error this framework keeps unless YOU drop it)")

ts = np.linspace(0, T, 400)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
plot_phase_space(mag_com.alpha_trajectory(ts)[:, :1], labels=["COM (ion 0)"], ax=ax)
plot_phase_space(mag_str.alpha_trajectory(ts)[:, :1], labels=["stretch (ion 0)"], ax=ax)
ax.set_title("flat pulse: circles, COM closes at T = 2$\\pi$/$\\delta$")
plt.show()

## 2. Detuning error → the loop doesn't close

Miscalibrate the detuning, $\delta \to \delta(1+\epsilon)$, but keep the *nominal* gate time
$T = 2\pi/\delta$. The trajectory is still a circle, but the walk around it is too fast:

$$ \alpha(T) = \frac{c}{\delta(1+\epsilon)}\left(e^{-i2\pi(1+\epsilon)} - 1\right) \neq 0. $$

The residual spin-dependent displacement leaves spin and motion entangled at the end of the gate;
tracing out the (unmeasured) mode then decoheres the spins — for each branch pair the coherence is
suppressed by the overlap $\langle \alpha_{\rm branch}|\alpha_{\rm branch}'\rangle \sim e^{-|\Delta\alpha|^2/2}$,
and the entangling angle is also slightly wrong. From here on we keep only the COM mode
(the stretch correction is the $\sim10^{-3}$ found above).

In [ ]:
eps = 0.30                          # deliberately large so everything is visible
mag_err = MSMagnus(b_com, eta, delta * (1 + eps), Omega, [0.0, 0.0], n_max)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
plot_phase_space(mag_com.alpha_trajectory(ts)[:, :1], labels=["ideal"], ax=ax)
plot_phase_space(mag_err.alpha_trajectory(ts)[:, :1], labels=[f"detuned {eps:+.0%}"], ax=ax)
ax.set_title("incorrect closure: the x marks where the gate ends")
plt.show()

# Bell infidelity vs detuning error (single mode, Magnus -- no ODE anywhere here)
psi0 = Operator(otimes(ket("00"), fock(0, n_max)))
bell1 = otimes((np.cos(chi) * ket("00") - 1j * np.sin(chi) * ket("11")), fock(0, n_max))
eps_scan = np.linspace(0, 0.3, 13)
infid = []
for e in eps_scan:
    m = MSMagnus(b_com, eta, delta * (1 + e), Omega, [0.0, 0.0], n_max)
    infid.append(1 - fidelity(bell1, np.asarray(m.unitary(T)) @ np.asarray(psi0)))
plt.semilogy(eps_scan, infid, "o-")
plt.xlabel("detuning error $\\epsilon$"); plt.ylabel("Bell infidelity $1-F$")
plt.title("gate error from imperfect loop closure")
plt.show()
print(f"alpha residual at eps={eps}: {abs(mag_err.alpha(T)[0]):.3f} (per ion)")

## 3. Same error, no Magnus: solve $H_{\rm RWA}(t)$ and look at the mode's Wigner function

Now evolve the **Hamiltonian itself** (`ms_lamb_dicke1(..., rwa=True)`, an ODE solve — first-order
Lamb–Dicke, RWA, but *no* closed form) with the same detuning error, and image the motional state.
The Wigner function of the reduced mode state,

$$ W(x,p) = \frac{1}{\pi}\int dy\; e^{-2ipy}\,\langle x+y|\rho_{\rm mode}|x-y\rangle,
\qquad \rho_{\rm mode} = \mathrm{Tr}_{q_0 q_1}\,|\psi(T)\rangle\langle\psi(T)|, $$

should sit exactly where the step-2 *trajectory* says it should: $|00\rangle$ is an equal
superposition of the four $\sigma_y$-branch states, displaced by $(s_0+s_1)\alpha(t)$,
$s_j = \pm1$ — so the spin-traced mode is the classical mixture
$\tfrac14|{+}2\alpha\rangle\langle\cdot| + \tfrac12|0\rangle\langle 0| + \tfrac14|{-}2\alpha\rangle\langle\cdot|$
(a mixture, hence no interference fringes; conditioning on a spin branch instead gives a single
displaced vacuum). Coherent state $|\alpha_0\rangle$ peaks at $(x,p) = \sqrt2(\mathrm{Re}\,\alpha_0, \mathrm{Im}\,\alpha_0)$.

**Under the hood:** `HamiltonianEvolution` flattens $\psi$ and hands $-iH(t)\psi$ to adaptive RK45
(rtol $10^{-8}$); nothing solves until `state_at` is called, and each call only extends the already
solved range. It prints every integration (silence with `htdse.quiet()`). The Magnus result and the
ODE agree on low-Fock states — they differ near the truncation edge, where the closed form is the
infinite-dimensional answer and the ODE propagates the truncated model.

In [ ]:
H_ld = ms_lamb_dicke1(b_com, eta, delta * (1 + eps), Omega, [0.0, 0.0], n_max, rwa=True)
ev = HamiltonianEvolution(H_ld, psi0)        # subsystems {"q0","q1","mode"} come from H_ld
psi_ode = ev.state_at(T)

# cross-check ODE vs Magnus on this (low-Fock) state
psi_mag = np.asarray(mag_err.unitary(T)) @ np.asarray(psi0)
print(f"|<Magnus|ODE>|^2 = {fidelity(psi_mag, psi_ode):.8f}")

rho_mode = ev.trace_out("q0", "q1", t=T)

xs = np.linspace(-3.2, 3.2, 241)
W = wigner(rho_mode, xs, xs)
fig, ax = plt.subplots(figsize=(6, 5.5))
lim = np.max(np.abs(W))
im = ax.pcolormesh(xs, xs, W, cmap="RdBu_r", vmin=-lim, vmax=lim, shading="auto")
fig.colorbar(im, ax=ax, label="W(x,p)")

# overlay the step-2 Magnus branch trajectories +-2 alpha(t) (in x,p units: sqrt(2)*alpha)
traj = 2 * mag_err.alpha_trajectory(ts)[:, 0]
for s, style in [(+1, "k-"), (-1, "k--")]:
    ax.plot(np.sqrt(2) * s * traj.real, np.sqrt(2) * s * traj.imag, style, lw=1.2,
            label=f"branch ${'+' if s > 0 else '-'}2\\alpha(t)$ (Magnus)")
    ax.plot(np.sqrt(2) * s * traj.real[-1], np.sqrt(2) * s * traj.imag[-1], "kx", ms=9)
ax.set_xlabel("x"); ax.set_ylabel("p"); ax.set_aspect("equal"); ax.legend(loc="upper left")
ax.set_title("spin-traced Wigner function vs the analytic trajectory")
plt.show()

# condition on the (+y,+y) spin branch: a single displaced vacuum exactly at +2 alpha(T)
_, vecs = np.linalg.eigh(sigma_y)
plus_y = vecs[:, 1]
mode_branch = np.einsum("i,j,ijk->k", plus_y.conj(), plus_y.conj(),
                        np.asarray(psi_ode).reshape(2, 2, n_max + 1))
mode_branch /= np.linalg.norm(mode_branch)
a_op = np.diag(np.sqrt(np.arange(1, n_max + 1)), 1)
print(f"<a> on (+y,+y) branch = {np.vdot(mode_branch, a_op @ mode_branch):+.4f}")
print(f"2*alpha(T) from Magnus = {2 * mag_err.alpha(T)[0]:+.4f}")

## 4. Imbalanced tones: asymmetric detuning

Let the blue and red tones be detuned *asymmetrically* from the sideband,
$\delta_{b} = \bar\delta + \epsilon_a$, $\delta_{r} = \bar\delta - \epsilon_a$. Summing the two tones,

$$ e^{-i\mu_b t} + e^{+i\mu_r t} = 2\,e^{-i\epsilon_a t}\cos(\bar\mu t), \qquad
\mu_{b,r} = \nu + \delta_{b,r},\ \bar\mu = \nu + \bar\delta, $$

i.e. the beat is unchanged but the **spin phase precesses**, $\phi_j(t) = \phi_j - \epsilon_a t$:
the direction of the spin-dependent force rotates in the equatorial plane during the gate.
Since $[\sigma_{\Phi(t_1)}, \sigma_{\Phi(t_2)}] \neq 0$, the Magnus series no longer terminates —
which is exactly why `MSMagnus` *refuses* time-dependent phases rather than returning a wrong
closed form. The LD builder takes callable phases, and the honest route is the ODE:

```python
ms_lamb_dicke1(b, eta, delta_bar, Omega, phases=[lambda t: -eps_a * t]*2, n_max, rwa=True)
```

In [ ]:
def imbalanced(eps_a):
    return ms_lamb_dicke1(b_com, eta, delta, Omega,
                          [lambda t, e=eps_a: -e * t, lambda t, e=eps_a: -e * t],
                          n_max, rwa=True)

# measured phase-space trajectory <a>(t) on the (+y,+y) branch, balanced vs imbalanced
plus_branch = Operator(otimes(plus_y, plus_y, fock(0, n_max)))
ts_meas = np.linspace(0, T, 120)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
with quiet():
    for eps_a, lbl in [(0.0, "balanced"), (0.15, "$\\epsilon_a=0.15$"), (0.3, "$\\epsilon_a=0.3$")]:
        ev_b = HamiltonianEvolution(imbalanced(eps_a), plus_branch)
        plot_phase_space(expectation_alpha(ev_b, ts_meas), labels=[lbl], ax=ax)
ax.set_title("force direction precesses: the circle shears open")
plt.show()

# Bell infidelity vs tone asymmetry (nominal T, correct mean detuning)
eps_a_scan = np.linspace(0, 0.3, 7)
infid_a = []
with quiet():
    for ea in eps_a_scan:
        psi_e = HamiltonianEvolution(imbalanced(ea), psi0).state_at(T)
        infid_a.append(1 - fidelity(bell1, psi_e))
plt.semilogy(eps_a_scan, np.maximum(infid_a, 1e-12), "o-")
plt.xlabel("tone asymmetry $\\epsilon_a$"); plt.ylabel("Bell infidelity $1-F$")
plt.title("gate error from asymmetric tones")
plt.show()
print("sanity: eps_a=0 infidelity =", f"{infid_a[0]:.2e}",
      "(matches the ideal Magnus gate up to stretch-free, ODE-tolerance level)")

## Numerical notes, collected

- **Steps 1–2 never solve an ODE.** `MSMagnus.unitary(t)` is one `expm` of the terminated Magnus
  generator; $\alpha,\Theta$ are trapezoid quadrature on a dense grid (`points_per_period=400`).
- **Steps 3–4 are real TDSE solves** of the term-layer $H_{\rm RWA}(t)$: lazy, extend-on-demand
  RK45 segments; each printed unless wrapped in `quiet()`. The mechanism is snapshot-frozen —
  mutating its parameters after building an evolution raises instead of reusing stale segments.
- **Fock truncation** (`n_max=12` here): branch displacements reach $|2\alpha| \approx 1$, so
  occupation $\sim|2\alpha|^2$ stays far below the edge. The Magnus/ODE cross-check in step 3 is
  performed on exactly such low-Fock states — full-space process fidelity would instead expose the
  (physically irrelevant) truncation-edge mismatch between the infinite-dimensional closed form and
  the truncated model.
- **Composability**: every $H$ here is a term-layer object, so `+ pauli_term("Z0", coeff=eps_z)`
  injects a $\sigma_z$ error, `TrotterizedMechanism(H, 0, T, n)` discretizes it, and
  `H.replace(sdf_q0=...)` swaps one ion's drive — same workflow as the other demos.